# Lab 5：Ascend C MatMul Tiling 与 Double Buffer 优化

## 实验目标

- 理解 `baseM`、`baseN`、`baseK` 与 `mTiles`、`nTiles`、`kTiles` 的关系。
- 能够在 Host 侧根据输入 Shape 计算 Tile 数量和 `usedCoreNum`。
- 能够在 Kernel 中完成输出 Tile 映射、GM 偏移计算和 K 维累加。
- 掌握 Double Buffer 的初始填充与稳态预取方法。
- 能够分阶段校验计算结果，并比较不同 Tiling 与 Buffer 配置的性能。

## 实验环境

| 项目 | 配置 |
| --- | --- |
| NPU | 单卡 Ascend 910B3（Atlas A2） |
| CANN | 9.0.0 |
| Python | 3.11.4 |
| 关键工具 | Ascend C、BiSheng、AscendCL、NumPy、pandas |

## 实验原理

本实验实现矩阵乘法 $C = AB$。Host 侧按 `baseM`、`baseN` 和 `baseK` 切分三个维度：

$$
mTiles=\frac{M}{baseM},\qquad
nTiles=\frac{N}{baseN},\qquad
kTiles=\frac{K}{baseK}.
$$

一个输出 Tile 对应一个 `(mTile, nTile)` 区域，所以输出 Tile 数量是 `mTiles * nTiles`。`kTiles` 属于同一输出区域的归约过程：第一块 K Tile 初始化 CO1，后续 K Tile 继续累加。Host 侧用输出 Tile 数量和可用 Cube Core 数计算 `usedCoreNum`，再用内核调用符把 GM 指针和基础整数参数传给 Kernel；Kernel 按 Core ID 跨步遍历输出 Tile。

矩阵在 GM 中使用 ND 布局。Kernel 先把 A、B 的 Tile 搬到 A1/B1，再转换到 A2/B2，调用 `Mmad` 累加，最后通过 `Fixpipe` 把 FP32 结果写回 C。B 按 16 列 Fragment 从 B1(NZ) 转为 B2(ZN)，与 Cube 的数据布局和 C0 粒度匹配。

Single Buffer 每次搬入一个 K Tile，计算完成后再复用缓冲区。Double Buffer 准备两个队列槽位，先填入前两个 K Tile；稳态阶段计算当前 Tile 的同时预取 `kTile + 2`。这样可以重叠搬运和计算，但队列、同步与片上存储也会增加开销，是否提速要以同一 Tiling 下的实测结果为准。

<img src="images/ascendc_matmul_pipeline.png" alt="Ascend C MatMul 流水" width="560">


In [ ]:
from pathlib import Path

import json
import math
import os
import platform
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


LAB_ROOT = Path.cwd().resolve()
OP_ROOT = (LAB_ROOT / "Sources" / "L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student").resolve()
HOST_ROOT = OP_ROOT / "op_host"
KERNEL_ROOT = OP_ROOT / "op_kernel"
RUNNER_ROOT = OP_ROOT / "runner"
INPUT_ROOT = OP_ROOT / "inputs"
RESULT_ROOT = OP_ROOT / "results"
RAW_ROOT = RESULT_ROOT / "raw"

IMAGE_ROOT = LAB_ROOT / "images"
assert IMAGE_ROOT.is_dir(), "未找到本实验的 images 目录。"

for path in [OP_ROOT, HOST_ROOT, KERNEL_ROOT, RUNNER_ROOT, INPUT_ROOT, RESULT_ROOT, RAW_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE_ID = int(os.environ.get("ASCEND_DEVICE_ID", "0"))
NPU_ARCH = os.environ.get("ASCEND_NPU_ARCH", "dav-2201")

VALIDATION_WARMUP = 1
VALIDATION_REPEAT = 2

# Validation progress controls:
# 1 = K Tiling only
# 2 = add M/N output Tiles
# 3 = add Core-stride validation
VALIDATION_LEVEL = 1

# 保持 False，直到完成 TODO 5/6 并通过 Level 3 Single Buffer 验证。
ENABLE_DOUBLE_BUFFER_VALIDATION = False

# Keep False until Level 3 Single/Double Buffer validation passes.
RUN_FULL_SWEEP = False
RESUME_SWEEP = False

SWEEP_WARMUP = 3
SWEEP_REPEAT = 10

RUNNER_MODE = "direct_kernel"
TIMING_SCHEMA = "direct_kernel_v2_0"
VALIDATION_VERSION = "STUDENT_DIRECT_KERNEL_VALIDATE_V2_0"
SWEEP_VERSION = "STUDENT_DIRECT_KERNEL_SWEEP_V2_0"

SHAPES = {
    "K_FIRST_ONLY": {"M": 32, "K": 64, "N": 32, "mode": "k_first_only", "atol": 1e-3, "rtol": 0.0},
    "K_SECOND_ONLY": {"M": 32, "K": 64, "N": 32, "mode": "k_second_only", "atol": 1e-3, "rtol": 0.0},
    "K_BOTH_SUM": {"M": 32, "K": 64, "N": 32, "mode": "k_both_sum", "atol": 2e-3, "rtol": 0.0},
    "K_RANDOM": {"M": 32, "K": 64, "N": 32, "mode": "random", "atol": 7e-3, "rtol": 2e-2},
    "MULTI_TILE_IDENTITY": {"M": 64, "K": 64, "N": 64, "mode": "identity", "atol": 1e-3, "rtol": 0.0},
    "MULTI_TILE_RANDOM": {
        "M": 64,
        "K": 64,
        "N": 64,
        "mode": "random",
        "atol": 1.2e-2,
        "rtol": 2e-2,
    },
    "CORE_STRIDE_RANDOM": {
        "M": 128,
        "K": 64,
        "N": 256,
        "mode": "random",
        "atol": 1.2e-2,
        "rtol": 2e-2,
    },
    "MAIN_BENCH": {"M": 128, "K": 2048, "N": 2048, "mode": "identity", "atol": 1e-3, "rtol": 0.0},
}

# T4 remains the already validated 32x32x32 baseline.
TILING_CANDIDATES = [
    {"id": "T1", "baseM": 16, "baseN": 16, "baseK": 16},
    {"id": "T2", "baseM": 32, "baseN": 16, "baseK": 32},
    {"id": "T3", "baseM": 16, "baseN": 32, "baseK": 32},
    {"id": "T4", "baseM": 32, "baseN": 32, "baseK": 32},
    {"id": "T5", "baseM": 64, "baseN": 64, "baseK": 64},
]

BUFFER_NUMS = [1, 2] if ENABLE_DOUBLE_BUFFER_VALIDATION else [1]

CONFIG = {
    "version": "student_direct_kernel_v2.0",
    "runner_mode": RUNNER_MODE,
    "timing_schema": TIMING_SCHEMA,
    "device_id": DEVICE_ID,
    "npu_arch": NPU_ARCH,
    "validation_warmup": VALIDATION_WARMUP,
    "validation_repeat": VALIDATION_REPEAT,
    "sweep_warmup": SWEEP_WARMUP,
    "sweep_repeat": SWEEP_REPEAT,
    "shapes": SHAPES,
    "tiling_candidates": TILING_CANDIDATES,
    "buffer_nums": BUFFER_NUMS,
}

(OP_ROOT / "experiment.json").write_text(
    json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("LAB_ROOT    :", LAB_ROOT)
print("OP_ROOT     :", OP_ROOT)
print("DEVICE_ID   :", DEVICE_ID)
print("NPU_ARCH    :", NPU_ARCH)
print("VALIDATION  : level", VALIDATION_LEVEL)
print("DOUBLE BUF  :", ENABLE_DOUBLE_BUFFER_VALIDATION)
print("FULL SWEEP  :", RUN_FULL_SWEEP)


## 实验流程

### 1. 初始化实验环境

初始化单元载入依赖，设置验证与性能实验参数，并创建 Host、Kernel、输入和结果目录。CANN 路径优先读取 `ASCEND_HOME_PATH`，未设置时再检查课程环境中的常用安装位置。


In [ ]:
def run_cmd(
    command,
    *,
    cwd: Path | None = None,
    env: dict | None = None,
    check: bool = True,
    capture: bool = False,
):
    print("+", " ".join(map(str, command)))
    return subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
        capture_output=capture,
    )


def find_cann_home() -> Path:
    candidates = [
        os.environ.get("ASCEND_HOME_PATH"),
        "/home/developer/Ascend/cann-9.0.0",
        "/usr/local/Ascend/ascend-toolkit/9.0.0",
        "/usr/local/Ascend/ascend-toolkit/latest",
    ]
    for value in candidates:
        if not value:
            continue
        path = Path(value).expanduser().resolve()
        if (path / "include").is_dir():
            return path
    raise FileNotFoundError("未找到 CANN 安装目录；请设置 ASCEND_HOME_PATH。")


CANN_HOME = find_cann_home()
os.environ["ASCEND_HOME_PATH"] = str(CANN_HOME)

checks = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "cann_home": str(CANN_HOME),
    "bisheng": shutil.which("bisheng"),
    "npu_smi": shutil.which("npu-smi"),
}

display(pd.DataFrame([{"item": key, "value": value} for key, value in checks.items()]))

assert checks["bisheng"], "bisheng 不在 PATH 中"

if checks["npu_smi"]:
    run_cmd(["npu-smi", "info"], check=False)


### 2. 构造可定位错误的输入

为了分别检查 K Offset 和 K 维累加，令

$$
A=[A_0\;A_1],\qquad A_0,A_1\in\mathbb{R}^{32\times32}.
$$

构造三种 B：

$$
B_{\mathrm{first}}=\begin{bmatrix}I\\0\end{bmatrix},\qquad
B_{\mathrm{second}}=\begin{bmatrix}0\\I\end{bmatrix},\qquad
B_{\mathrm{sum}}=\begin{bmatrix}I\\I\end{bmatrix}.
$$

三组输出分别是 `A0`、`A1` 和 `A0 + A1`。如果其中一组出错，可以据此区分 K Tile 偏移和累加问题。

In [ ]:
def make_structured_a(M: int, K: int) -> np.ndarray:
    """Distinct FP16 values; multiples of 1/16 are exactly representable."""
    rows = np.arange(M, dtype=np.int32)[:, None]
    cols = np.arange(K, dtype=np.int32)[None, :]

    values = (((17 * rows + 7 * cols) % 61) - 30) / 16.0

    return values.astype(np.float16)


def make_random_matrix(shape: tuple[int, int], seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return (rng.standard_normal(shape) * 0.05).astype(np.float16)


def write_case(shape_id: str, spec: dict) -> dict:
    M = spec["M"]
    K = spec["K"]
    N = spec["N"]
    mode = spec["mode"]

    if mode in {"k_first_only", "k_second_only", "k_both_sum"}:
        assert M == N == 32
        assert K == 64

        a = make_structured_a(M, K)
        b = np.zeros((K, N), dtype=np.float16)
        identity = np.eye(32, dtype=np.float16)

        if mode in {"k_first_only", "k_both_sum"}:
            b[:32, :] = identity

        if mode in {"k_second_only", "k_both_sum"}:
            b[32:, :] = identity

    elif mode == "identity":
        if K != N:
            raise ValueError(f"identity mode requires K == N, got K={K}, N={N}")

        a = make_random_matrix((M, K), seed=20260807 + M + 3 * K)
        b = np.eye(K, dtype=np.float16)

    elif mode == "random":
        seed = 20260807 + M + 3 * K + 7 * N
        a = make_random_matrix((M, K), seed=seed)
        b = make_random_matrix((K, N), seed=seed + 1)

    else:
        raise ValueError(f"unsupported mode: {mode}")

    if mode == "identity":
        golden = a.astype(np.float32)
    else:
        golden = a.astype(np.float32) @ b.astype(np.float32)

    if mode == "k_first_only":
        assert np.array_equal(golden, a[:, :32].astype(np.float32))
    elif mode == "k_second_only":
        assert np.array_equal(golden, a[:, 32:].astype(np.float32))
    elif mode == "k_both_sum":
        expected = a[:, :32].astype(np.float32) + a[:, 32:].astype(np.float32)
        assert np.array_equal(golden, expected)

    case_dir = INPUT_ROOT / shape_id
    case_dir.mkdir(parents=True, exist_ok=True)

    a.tofile(case_dir / "A.bin")
    b.tofile(case_dir / "B.bin")
    golden.tofile(case_dir / "C_golden.bin")
    np.save(case_dir / "C_golden.npy", golden)

    record = {
        "shape_id": shape_id,
        "M": M,
        "K": K,
        "N": N,
        "mode": mode,
        "a_mib": float(a.nbytes / 1024**2),
        "b_mib": float(b.nbytes / 1024**2),
        "c_mib": float(golden.nbytes / 1024**2),
        "golden_max_abs": float(np.max(np.abs(golden))),
        "atol": spec["atol"],
        "rtol": spec["rtol"],
    }

    (case_dir / "meta.json").write_text(
        json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    return record


input_table = pd.DataFrame([write_case(shape_id, spec) for shape_id, spec in SHAPES.items()])

display(input_table)

### 3. 配置 Tiling 与片上 Buffer

下一单元定义 T1–T5 五组 `baseM / baseN / baseK`，并估算 A1、B1、A2、B2 和 CO1 的占用。输出 Tile 只由 M、N 两个方向决定；K Tile 在每个输出区域内部累加。

<img src="images/matmul_tiling_design.png" alt="MatMul Tiling 设计" width="560">

本实验固定 `fragmentN = 16`。B1 使用 NZ 布局，按 16 列转换到 B2 的 ZN 布局，再执行 `Mmad` 和 `Fixpipe`。

In [ ]:
def get_candidate(tiling_id: str) -> dict:
    for item in TILING_CANDIDATES:
        if item["id"] == tiling_id:
            return item
    raise KeyError(tiling_id)


def estimate_candidate(shape: dict, candidate: dict, buffer_num: int) -> dict:
    M = shape["M"]
    K = shape["K"]
    N = shape["N"]

    bm = candidate["baseM"]
    bn = candidate["baseN"]
    bk = candidate["baseK"]

    valid = (
        buffer_num in {1, 2}
        and bm % 16 == 0
        and bn % 16 == 0
        and bk % 16 == 0
        and M % bm == 0
        and N % bn == 0
        and K % bk == 0
    )

    m_tiles = M // bm if valid else None
    n_tiles = N // bn if valid else None
    k_tiles = K // bk if valid else None

    a1_bytes = buffer_num * bm * bk * 2
    b1_bytes = buffer_num * bk * bn * 2
    a2_bytes = bm * bk * 2
    b2_bytes = bk * 16 * 2
    co1_bytes = bm * 16 * 4

    return {
        "tiling_id": candidate["id"],
        "baseM": bm,
        "baseN": bn,
        "baseK": bk,
        "buffer_num": buffer_num,
        "valid": valid,
        "mTiles": m_tiles,
        "nTiles": n_tiles,
        "kTiles": k_tiles,
        "outputTileCount": (m_tiles * n_tiles if valid else None),
        "nFragments": (bn // 16 if valid else None),
        "A1_bytes": a1_bytes,
        "B1_bytes": b1_bytes,
        "A2_bytes": a2_bytes,
        "B2_bytes": b2_bytes,
        "CO1_bytes": co1_bytes,
        "CO2_bytes": 0,
        "estimated_total_bytes": (a1_bytes + b1_bytes + a2_bytes + b2_bytes + co1_bytes),
    }


candidate_table = pd.DataFrame(
    [
        estimate_candidate(SHAPES["MAIN_BENCH"], candidate, buffer_num)
        for candidate in TILING_CANDIDATES
        for buffer_num in BUFFER_NUMS
    ]
)

display(candidate_table)

assert candidate_table["valid"].all()
assert (candidate_table["CO2_bytes"] == 0).all()

### 4. 组织 Host 与 Kernel 源码

本实验不生成 OPP 工程。`op_host` 负责根据 Shape 与可用 Cube Core 构造 Tiling 信息，`op_kernel` 负责计算；Runner 将两份源码包含到同一个 `.asc` 文件，由 BiSheng 编译。

Kernel 的调用边界固定为三个 GM 地址与七个 `uint32_t` 参数：`M / N / K / mTiles / nTiles / kTiles / usedCoreNum`。Host 侧的 Tiling 结构只在进程内使用，不会按值传入 Kernel。


In [ ]:
HOST_SOURCE = HOST_ROOT / "cube_matmul_custom.cpp"
KERNEL_SOURCE = KERNEL_ROOT / "cube_matmul_custom.cpp"
RUNNER_SOURCE = RUNNER_ROOT / "matmul_benchmark.asc"
RUNNER_BINARY = RUNNER_ROOT / "matmul_benchmark"

print("Host source  :", HOST_SOURCE)
print("Kernel source:", KERNEL_SOURCE)
print("Runner source:", RUNNER_SOURCE)


In [ ]:
# 这份实验直接编译 Runner，不执行工程生成、OPP 打包或安装。


### 5. 定义 Host 侧 Tiling 与编译配置

Host 在进程内保存 `M / N / K`、`mTiles / nTiles / kTiles` 与 `usedCoreNum`，并在启动 Kernel 时逐项传入。`experiment_config.h` 固定本次编译使用的 `LAB_BASE_*` 与 Buffer 模式；Kernel 从编译期配置读取 Tile 大小。


In [ ]:
# 不再生成或传递 TilingData 缓冲区。Host 侧结构定义在下一节的 op_host 源码中。


In [ ]:
def experiment_config_text(candidate: dict, buffer_num: int) -> str:
    return (
        "#ifndef EXPERIMENT_CONFIG_H\n"
        "#define EXPERIMENT_CONFIG_H\n"
        "\n"
        f"#define LAB_BASE_M {candidate['baseM']}\n"
        f"#define LAB_BASE_N {candidate['baseN']}\n"
        f"#define LAB_BASE_K {candidate['baseK']}\n"
        f"#define LAB_BUFFER_NUM {buffer_num}\n"
        "\n"
        "#endif\n"
    )


### 6. 补全 Host 侧 Tiling

Host 根据 `M / N / K` 计算 `mTiles`、`nTiles` 和 `kTiles`，再用输出 Tile 数量与可用 Cube Core 数确定 `usedCoreNum`。完成下一单元中的 TODO 1；生成的 Tiling 仅用于设置启动核数和逐项传递 Kernel 参数。


In [ ]:
%%writefile Sources/L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student/op_host/cube_matmul_custom.cpp

#include "experiment_config.h"

#include <algorithm>
#include <cstdint>

struct CubeMatmulCustomTiling {
    uint32_t M = 0;
    uint32_t N = 0;
    uint32_t K = 0;
    uint32_t mTiles = 0;
    uint32_t nTiles = 0;
    uint32_t kTiles = 0;
    uint32_t usedCoreNum = 0;
};

inline bool BuildCubeMatmulCustomTiling(uint32_t M, uint32_t N, uint32_t K,
                                        uint32_t cubeCoreNum,
                                        CubeMatmulCustomTiling* tiling)
{
    if (tiling == nullptr || cubeCoreNum == 0 || M == 0 || N == 0 || K == 0) {
        return false;
    }

    if (LAB_BASE_M % 16 != 0 || LAB_BASE_N % 16 != 0 || LAB_BASE_K % 16 != 0 ||
        M % LAB_BASE_M != 0 || N % LAB_BASE_N != 0 || K % LAB_BASE_K != 0) {
        return false;
    }

    // TODO(student 1):
    // Compute the number of Tiles along M, N and K.
    //
    // Placeholder values intentionally make the validation fail
    // until this TODO is completed.
    const uint32_t mTiles = 0;
    const uint32_t nTiles = 0;
    const uint32_t kTiles = 0;

    const uint32_t outputTileCount = mTiles * nTiles;
    const uint32_t usedCoreNum = std::max(1U, std::min(outputTileCount, cubeCoreNum));

    tiling->M = M;
    tiling->N = N;
    tiling->K = K;
    tiling->mTiles = mTiles;
    tiling->nTiles = nTiles;
    tiling->kTiles = kTiles;
    tiling->usedCoreNum = usedCoreNum;
    return true;
}


### 7. 补全 Device Kernel

Kernel 接收 Host 逐项传入的 Shape 与 Tiling 参数，映射输出 Tile，计算 A、B、C 在 GM 中的偏移，并沿 K 维累加。代码中有 5 处待补全内容：

- TODO 2：输出 Tile 映射；
- TODO 3：A、B、C 的 GM 偏移；
- TODO 4：K Tile 在 CO1 中的累加方式；
- TODO 5：A1/B1 队列深度；
- TODO 6：Double Buffer 的初始填充与稳态预取。

先在 Single Buffer 下完成 TODO 2–4 并通过验证，再处理 TODO 5–6。

<img src="images/double_buffer_pipeline.png" alt="Double Buffer 流水" width="560">

这份教学实现把重点放在 Tiling 和 Double Buffer。每个 16 列 Fragment 都会重新搬运完整 A Tile 和当前 `baseN` 宽度的 B Tile，没有跨 N Fragment 复用 A1/A2，也没有把 B 的 GM 搬运缩小到单个 Fragment。分析性能时要把这部分开销考虑进去。


In [ ]:
%%writefile Sources/L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student/op_kernel/cube_matmul_custom.cpp

#include "kernel_operator.h"
#include "experiment_config.h"

using namespace AscendC;

template <int32_t BUFFER_NUM>
class KernelMmadTiled {
private:
    // TODO(student 5):
    // Replace the fixed value with the compile-time Buffer mode.
    static constexpr int32_t A1B1_QUEUE_DEPTH = 1;

public:
    __aicore__ inline KernelMmadTiled() {}

    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR c, uint32_t M, uint32_t N, uint32_t K,
                                uint32_t baseM, uint32_t baseN, uint32_t baseK, uint32_t mTiles,
                                uint32_t nTiles, uint32_t kTiles, uint32_t usedCoreNum)
    {
        M_ = M;
        N_ = N;
        K_ = K;

        baseM_ = baseM;
        baseN_ = baseN;
        baseK_ = baseK;

        mTiles_ = mTiles;
        nTiles_ = nTiles;
        kTiles_ = kTiles;
        usedCoreNum_ = usedCoreNum;

        mBlocks_ = baseM_ / C0;
        kBlocks_ = baseK_ / C0;
        nFragments_ = baseN_ / FRAGMENT_N;

        aGm_.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(a), M_ * K_);
        bGm_.SetGlobalBuffer(reinterpret_cast<__gm__ half*>(b), K_ * N_);
        cGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(c), M_ * N_);

        const uint32_t aTileElements = baseM_ * baseK_;
        const uint32_t bTileElements = baseK_ * baseN_;
        const uint32_t bFragmentElements = baseK_ * FRAGMENT_N;
        const uint32_t cFragmentElements = baseM_ * FRAGMENT_N;

        // TODO(student 5):
        // A1/B1 queue depth must match Single/Double Buffer mode.
        pipe_.InitBuffer(a1Queue_, A1B1_QUEUE_DEPTH, aTileElements * sizeof(half));
        pipe_.InitBuffer(b1Queue_, A1B1_QUEUE_DEPTH, bTileElements * sizeof(half));

        pipe_.InitBuffer(a2Queue_, 1, aTileElements * sizeof(half));
        pipe_.InitBuffer(b2Queue_, 1, bFragmentElements * sizeof(half));
        pipe_.InitBuffer(c1Queue_, 1, cFragmentElements * sizeof(float));
    }

    __aicore__ inline void Process()
    {
        const uint32_t outputTileCount = mTiles_ * nTiles_;

        for (uint32_t outputTile = GetBlockIdx(); outputTile < outputTileCount;
             outputTile += usedCoreNum_) {
            // TODO(student 2):
            // Map one-dimensional outputTile to (mTile, nTile).
            const uint32_t mTile = 0;
            const uint32_t nTile = 0;

            ComputeOutputTile(mTile, nTile);
        }
    }

private:
    __aicore__ inline void CopyIn(uint32_t mTile, uint32_t nTile, uint32_t kTile)
    {
        LocalTensor<half> a1 = a1Queue_.template AllocTensor<half>();
        LocalTensor<half> b1 = b1Queue_.template AllocTensor<half>();

        const uint32_t mStart = mTile * baseM_;
        const uint32_t nStart = nTile * baseN_;
        const uint32_t kStart = kTile * baseK_;

        // TODO(student 3):
        // Compute A/B offsets in row-major GM(ND).
        const uint32_t aOffset = 0;
        const uint32_t bOffset = 0;

        Nd2NzParams aParams{};
        aParams.ndNum = 1;
        aParams.nValue = baseM_;
        aParams.dValue = baseK_;
        aParams.srcNdMatrixStride = 0;
        aParams.srcDValue = K_;
        aParams.dstNzC0Stride = baseM_;
        aParams.dstNzNStride = 1;
        aParams.dstNzMatrixStride = 0;

        Nd2NzParams bParams{};
        bParams.ndNum = 1;
        bParams.nValue = baseK_;
        bParams.dValue = baseN_;
        bParams.srcNdMatrixStride = 0;
        bParams.srcDValue = N_;
        bParams.dstNzC0Stride = baseK_;
        bParams.dstNzNStride = 1;
        bParams.dstNzMatrixStride = 0;

        DataCopy(a1, aGm_[aOffset], aParams);
        DataCopy(b1, bGm_[bOffset], bParams);

        a1Queue_.EnQue(a1);
        b1Queue_.EnQue(b1);
    }

    __aicore__ inline void SplitA(const LocalTensor<half>& a1)
    {
        LocalTensor<half> a2 = a2Queue_.template AllocTensor<half>();

        uint32_t srcOffset = 0;
        uint32_t dstOffset = 0;

        for (uint16_t mBlock = 0; mBlock < mBlocks_; ++mBlock) {
            LoadData2DParams params{};
            params.repeatTimes = kBlocks_;
            params.srcStride = mBlocks_;
            params.ifTranspose = false;

            LoadData(a2[dstOffset], a1[srcOffset], params);

            srcOffset += C0 * C0;
            dstOffset += baseK_ * C0;
        }

        a2Queue_.EnQue(a2);
    }

    __aicore__ inline void SplitB(const LocalTensor<half>& b1, uint16_t fragment)
    {
        LocalTensor<half> b2 = b2Queue_.template AllocTensor<half>();

        LoadData2DParams params{};
        params.repeatTimes = kBlocks_;
        params.srcStride = 1;
        params.ifTranspose = true;

        const uint32_t srcOffset = static_cast<uint32_t>(fragment) * baseK_ * FRAGMENT_N;

        LoadData(b2, b1[srcOffset], params);

        b2Queue_.EnQue(b2);
    }

    __aicore__ inline void MmadKTile(const LocalTensor<float>& c1, const LocalTensor<half>& a2,
                                     const LocalTensor<half>& b2, uint32_t kTile)
    {
        MmadParams params{};
        params.m = baseM_;
        params.n = FRAGMENT_N;
        params.k = baseK_;
        params.cmatrixSource = false;

        // TODO(student 4):
        // The first K Tile starts from zero.
        // Later K Tiles must accumulate on the existing CO1 result.
        //
        // The placeholder below reinitializes C every time, so
        // K accumulation tests will fail until it is corrected.
        params.cmatrixInitVal = true;

        Mmad(c1, a2, b2, params);

        PipeBarrier<PIPE_M>();
    }

    __aicore__ inline void CopyOut(const LocalTensor<float>& c1, uint32_t mTile, uint32_t nTile,
                                   uint16_t fragment)
    {
        const uint32_t mStart = mTile * baseM_;
        const uint32_t nStart = nTile * baseN_;

        // TODO(student 3):
        // Compute the row-major C offset for this output Tile
        // and this 16-column fragment.
        const uint32_t cOffset = 0;

        FixpipeParamsV220 params{};
        params.nSize = FRAGMENT_N;
        params.mSize = baseM_;
        params.srcStride = baseM_;
        params.dstStride = N_;
        params.ndNum = 1;
        params.srcNdStride = 0;
        params.dstNdStride = 0;
        params.quantPre = QuantMode_t::NoQuant;

        Fixpipe(cGm_[cOffset], c1, params);
    }

    __aicore__ inline void ComputeFragment(uint32_t mTile, uint32_t nTile, uint16_t fragment)
    {
        LocalTensor<float> c1 = c1Queue_.template AllocTensor<float>();

        CopyIn(mTile, nTile, 0);

        if (BUFFER_NUM == 2 && kTiles_ > 1) {
            // TODO(student 6):
            // Fill the second A1/B1 slot before entering the loop.
        }

        for (uint32_t kTile = 0; kTile < kTiles_; ++kTile) {
            LocalTensor<half> a1 = a1Queue_.template DeQue<half>();
            LocalTensor<half> b1 = b1Queue_.template DeQue<half>();

            SplitA(a1);
            SplitB(b1, fragment);

            a1Queue_.FreeTensor(a1);
            b1Queue_.FreeTensor(b1);

            // TODO(student 6):
            // After A1/B1 for the current Tile are freed,
            // prefetch the future K Tile that keeps two slots full.
            if (BUFFER_NUM == 2) {
                // Add the boundary condition and CopyIn call here.
            }

            LocalTensor<half> a2 = a2Queue_.template DeQue<half>();
            LocalTensor<half> b2 = b2Queue_.template DeQue<half>();

            MmadKTile(c1, a2, b2, kTile);

            a2Queue_.FreeTensor(a2);
            b2Queue_.FreeTensor(b2);

            if (BUFFER_NUM == 1 && kTile + 1 < kTiles_) {
                CopyIn(mTile, nTile, kTile + 1);
            }
        }

        c1Queue_.EnQue(c1);
        LocalTensor<float> c1Ready = c1Queue_.template DeQue<float>();

        CopyOut(c1Ready, mTile, nTile, fragment);

        c1Queue_.FreeTensor(c1Ready);
    }

    __aicore__ inline void ComputeOutputTile(uint32_t mTile, uint32_t nTile)
    {
        for (uint16_t fragment = 0; fragment < nFragments_; ++fragment) {
            ComputeFragment(mTile, nTile, fragment);
        }
    }

private:
    static constexpr uint16_t C0 = 16;
    static constexpr uint16_t FRAGMENT_N = 16;

    TPipe pipe_;

    TQue<TPosition::A1, A1B1_QUEUE_DEPTH> a1Queue_;
    TQue<TPosition::B1, A1B1_QUEUE_DEPTH> b1Queue_;
    TQue<TPosition::A2, 1> a2Queue_;
    TQue<TPosition::B2, 1> b2Queue_;
    TQue<TPosition::CO1, 1> c1Queue_;

    GlobalTensor<half> aGm_;
    GlobalTensor<half> bGm_;
    GlobalTensor<float> cGm_;

    uint32_t M_ = 0;
    uint32_t N_ = 0;
    uint32_t K_ = 0;

    uint32_t baseM_ = 0;
    uint32_t baseN_ = 0;
    uint32_t baseK_ = 0;

    uint32_t mTiles_ = 0;
    uint32_t nTiles_ = 0;
    uint32_t kTiles_ = 0;
    uint32_t usedCoreNum_ = 0;

    uint16_t mBlocks_ = 0;
    uint16_t kBlocks_ = 0;
    uint16_t nFragments_ = 0;
};

extern "C" __global__ __aicore__ void cube_matmul_custom(
    GM_ADDR a, GM_ADDR b, GM_ADDR c, uint32_t M, uint32_t N, uint32_t K,
    uint32_t mTiles, uint32_t nTiles, uint32_t kTiles, uint32_t usedCoreNum)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIC_ONLY);

    if (M == 0 || N == 0 || K == 0 || mTiles == 0 || nTiles == 0 || kTiles == 0 ||
        usedCoreNum == 0 || M % LAB_BASE_M != 0 || N % LAB_BASE_N != 0 ||
        K % LAB_BASE_K != 0 || mTiles != M / LAB_BASE_M || nTiles != N / LAB_BASE_N ||
        kTiles != K / LAB_BASE_K) {
        return;
    }

    KernelMmadTiled<LAB_BUFFER_NUM> kernel;
    kernel.Init(a, b, c, M, N, K, LAB_BASE_M, LAB_BASE_N, LAB_BASE_K, mTiles, nTiles, kTiles,
                usedCoreNum);
    kernel.Process();
}


### 8. 准备每个 Tiling 变体

每个 `(Tiling, BUFFER_NUM)` 组合都会把 `LAB_BASE_*` 和 `LAB_BUFFER_NUM` 写入 Host、Kernel 各自目录下的 `experiment_config.h`。不再构建、安装或加载 OPP 包。


In [ ]:
def write_with_magic(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    get_ipython().run_cell_magic("writefile", str(path), content)


def prepare_variant(tiling_id: str, buffer_num: int) -> dict:
    candidate = get_candidate(tiling_id)
    config = experiment_config_text(candidate, buffer_num)

    write_with_magic(HOST_ROOT / "experiment_config.h", config)
    write_with_magic(KERNEL_ROOT / "experiment_config.h", config)

    required_sources = [HOST_SOURCE, KERNEL_SOURCE, RUNNER_SOURCE]
    missing_sources = [str(path) for path in required_sources if not path.is_file()]
    if missing_sources:
        raise FileNotFoundError("请先执行所有源码 %%writefile 单元：" + ", ".join(missing_sources))

    return candidate


### 9. 编写并编译 Benchmark Runner

Runner 使用 AscendCL 分配与复制 GM 内存，查询可用 Cube Core，再通过字面量 `<<<...>>>` 启动 Kernel。计时只覆盖 Kernel launch 到 Stream 同步的区间；CSV 写入 `direct_kernel_v2_0` schema，避免与旧测量混用。


In [ ]:
%%writefile Sources/L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student/runner/matmul_benchmark.asc

#include <acl/acl.h>

#include <algorithm>
#include <cstdint>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

#include "../op_kernel/cube_matmul_custom.cpp"
#include "../op_host/cube_matmul_custom.cpp"

constexpr const char* kTimingSchema = "direct_kernel_v2_0";
constexpr const char* kLaunchMode = "direct_kernel";

static void PrintRecentAclError()
{
    const char* message = aclGetRecentErrMsg();
    std::cerr << "aclGetRecentErrMsg: "
              << ((message != nullptr && message[0] != '\0') ? message : "<empty>") << std::endl;
}

#define CHECK_ACL(expr)                                                                  \
    do {                                                                                 \
        const aclError status = (expr);                                                 \
        if (status != ACL_SUCCESS) {                                                    \
            std::cerr << "ACL failure: " << #expr << ", status=" << status << std::endl; \
            PrintRecentAclError();                                                      \
            std::exit(1);                                                               \
        }                                                                                \
    } while (0)

template <typename T>
std::vector<T> ReadBinary(const std::string& path, size_t count)
{
    std::vector<T> data(count);
    std::ifstream input(path, std::ios::binary);
    if (!input) {
        throw std::runtime_error("cannot open input: " + path);
    }
    input.read(reinterpret_cast<char*>(data.data()),
               static_cast<std::streamsize>(count * sizeof(T)));
    if (!input) {
        throw std::runtime_error("cannot read input: " + path);
    }
    return data;
}

template <typename T>
void WriteBinary(const std::string& path, const std::vector<T>& data)
{
    std::ofstream output(path, std::ios::binary);
    if (!output) {
        throw std::runtime_error("cannot open output: " + path);
    }
    output.write(reinterpret_cast<const char*>(data.data()),
                 static_cast<std::streamsize>(data.size() * sizeof(T)));
    if (!output) {
        throw std::runtime_error("cannot write output: " + path);
    }
}

uint32_t ParseDimension(const char* text, const char* name)
{
    char* end = nullptr;
    const unsigned long long value = std::strtoull(text, &end, 10);
    if (text == end || *end != '\0' || value == 0 || value > UINT32_MAX) {
        throw std::runtime_error(std::string("invalid ") + name + ": " + text);
    }
    return static_cast<uint32_t>(value);
}

int ParseDeviceId(const char* text)
{
    char* end = nullptr;
    const unsigned long long value = std::strtoull(text, &end, 10);
    if (text == end || *end != '\0' || value > static_cast<unsigned long long>(std::numeric_limits<int>::max())) {
        throw std::runtime_error(std::string("invalid DEVICE: ") + text);
    }
    return static_cast<int>(value);
}

uint32_t QueryCubeCoreNum(int deviceId)
{
    int64_t cubeCoreNum = 0;
    CHECK_ACL(aclrtGetDeviceInfo(static_cast<uint32_t>(deviceId), ACL_DEV_ATTR_CUBE_CORE_NUM,
                                 &cubeCoreNum));
    if (cubeCoreNum <= 0 || cubeCoreNum > UINT32_MAX) {
        throw std::runtime_error("device reports an invalid Cube Core count");
    }
    return static_cast<uint32_t>(cubeCoreNum);
}

int main(int argc, char** argv)
{
    if (argc != 12) {
        std::cerr << "usage: runner DEVICE M K N A B C WARMUP REPEAT TIMES_CSV LABEL" << std::endl;
        return 2;
    }

    const int deviceId = ParseDeviceId(argv[1]);
    const uint32_t M = ParseDimension(argv[2], "M");
    const uint32_t K = ParseDimension(argv[3], "K");
    const uint32_t N = ParseDimension(argv[4], "N");
    const std::string aPath = argv[5];
    const std::string bPath = argv[6];
    const std::string cPath = argv[7];
    const uint32_t warmup = ParseDimension(argv[8], "WARMUP");
    const uint32_t repeat = ParseDimension(argv[9], "REPEAT");
    const std::string timesPath = argv[10];
    const std::string label = argv[11];

    const size_t aCount = static_cast<size_t>(M) * K;
    const size_t bCount = static_cast<size_t>(K) * N;
    const size_t cCount = static_cast<size_t>(M) * N;
    const size_t aBytes = aCount * sizeof(aclFloat16);
    const size_t bBytes = bCount * sizeof(aclFloat16);
    const size_t cBytes = cCount * sizeof(float);

    auto aHost = ReadBinary<aclFloat16>(aPath, aCount);
    auto bHost = ReadBinary<aclFloat16>(bPath, bCount);
    constexpr float kOutputSentinel = 123.0F;
    std::vector<float> cHost(cCount, kOutputSentinel);

    CHECK_ACL(aclInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));

    aclrtStream stream = nullptr;
    uint8_t* aDevice = nullptr;
    uint8_t* bDevice = nullptr;
    uint8_t* cDevice = nullptr;
    aclrtEvent start = nullptr;
    aclrtEvent end = nullptr;

    CHECK_ACL(aclrtCreateStream(&stream));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&aDevice), aBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&bDevice), bBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMalloc(reinterpret_cast<void**>(&cDevice), cBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL(aclrtMemcpy(aDevice, aBytes, aHost.data(), aBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(bDevice, bBytes, bHost.data(), bBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(cDevice, cBytes, cHost.data(), cBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    const uint32_t cubeCoreNum = QueryCubeCoreNum(deviceId);
    CubeMatmulCustomTiling tiling{};
    if (!BuildCubeMatmulCustomTiling(M, N, K, cubeCoreNum, &tiling)) {
        std::cerr << "unsupported shape or tiling configuration" << std::endl;
        return 3;
    }

    std::cerr << "launch_mode=" << kLaunchMode << " M=" << M << " N=" << N << " K=" << K
              << " mTiles=" << tiling.mTiles << " nTiles=" << tiling.nTiles
              << " kTiles=" << tiling.kTiles << " blockDim=" << tiling.usedCoreNum << std::endl;

    for (uint32_t i = 0; i < warmup; ++i) {
        cube_matmul_custom<<<tiling.usedCoreNum, nullptr, stream>>>(
            aDevice, bDevice, cDevice, tiling.M, tiling.N, tiling.K, tiling.mTiles, tiling.nTiles,
            tiling.kTiles, tiling.usedCoreNum);
    }
    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtCreateEvent(&start));
    CHECK_ACL(aclrtCreateEvent(&end));

    std::vector<float> times;
    times.reserve(repeat);
    for (uint32_t i = 0; i < repeat; ++i) {
        CHECK_ACL(aclrtRecordEvent(start, stream));
        cube_matmul_custom<<<tiling.usedCoreNum, nullptr, stream>>>(
            aDevice, bDevice, cDevice, tiling.M, tiling.N, tiling.K, tiling.mTiles, tiling.nTiles,
            tiling.kTiles, tiling.usedCoreNum);
        CHECK_ACL(aclrtRecordEvent(end, stream));
        CHECK_ACL(aclrtSynchronizeStream(stream));

        float milliseconds = 0.0F;
        CHECK_ACL(aclrtEventElapsedTime(&milliseconds, start, end));
        times.push_back(milliseconds * 1000.0F);
    }

    CHECK_ACL(aclrtMemcpy(cHost.data(), cBytes, cDevice, cBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    WriteBinary(cPath, cHost);

    std::ofstream timesFile(timesPath);
    if (!timesFile) {
        throw std::runtime_error("cannot open timing CSV: " + timesPath);
    }
    timesFile << "timing_schema,launch_mode,label,iteration,time_us\n";
    for (uint32_t i = 0; i < repeat; ++i) {
        timesFile << kTimingSchema << "," << kLaunchMode << "," << label << "," << i << ","
                  << times[i] << "\n";
    }

    CHECK_ACL(aclrtDestroyEvent(start));
    CHECK_ACL(aclrtDestroyEvent(end));
    CHECK_ACL(aclrtFree(aDevice));
    CHECK_ACL(aclrtFree(bDevice));
    CHECK_ACL(aclrtFree(cDevice));
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}


In [ ]:
if not RUNNER_SOURCE.is_file():
    raise FileNotFoundError("请先执行 Runner 的 %%writefile 单元。")

print(RUNNER_SOURCE)


In [ ]:
def compile_runner(tiling_id: str, buffer_num: int) -> dict:
    candidate = prepare_variant(tiling_id, buffer_num)
    command = [
        "bisheng",
        "-O2",
        RUNNER_SOURCE,
        "-o",
        RUNNER_BINARY,
        f"--npu-arch={NPU_ARCH}",
        "-I",
        CANN_HOME / "include",
        "-L",
        CANN_HOME / "lib64",
        f"-Wl,-rpath,{CANN_HOME / 'lib64'}",
        "-lascendcl",
        "-lruntime",
    ]
    run_cmd(command)
    return {"candidate": candidate, "binary": RUNNER_BINARY}


### 10. 定义正确性与性能指标

正确性以逐元素 `np.allclose(atol, rtol)` 为主要判据，同时要求 `actual` 全部为有限值、`NRMSE <= 0.05` 且 `sentinel_count == 0`。`max_abs_error`、RMSE 和 NRMSE 用于定位误差。

性能记录中位耗时 `median_us`、P90、CV 和 GFLOP/s。比较配置时以中位耗时为主，P90 与 CV 用于判断波动。计时 CSV 必须带有 `direct_kernel_v2_0` schema 和 `direct_kernel` 启动方式。


In [ ]:
def output_path(version: str, shape_id: str, tiling_id: str, buffer_num: int) -> Path:
    name = f"{version}__{shape_id}__{tiling_id}__B{buffer_num}.bin"
    return RAW_ROOT / name


def times_path(version: str, shape_id: str, tiling_id: str, buffer_num: int) -> Path:
    name = f"{version}__{shape_id}__{tiling_id}__B{buffer_num}.csv"
    return RAW_ROOT / name


def run_variant(
    *,
    version: str,
    shape_id: str,
    tiling_id: str,
    buffer_num: int,
    warmup: int,
    repeat: int,
) -> dict:
    spec = SHAPES[shape_id]
    case_dir = INPUT_ROOT / shape_id
    out = output_path(version, shape_id, tiling_id, buffer_num)
    times = times_path(version, shape_id, tiling_id, buffer_num)

    run_cmd(
        [
            RUNNER_BINARY,
            DEVICE_ID,
            spec["M"],
            spec["K"],
            spec["N"],
            case_dir / "A.bin",
            case_dir / "B.bin",
            out,
            warmup,
            repeat,
            times,
            version,
        ]
    )

    return {
        "version": version,
        "runner_mode": RUNNER_MODE,
        "timing_schema": TIMING_SCHEMA,
        "shape_id": shape_id,
        "tiling_id": tiling_id,
        "buffer_num": buffer_num,
        "output": out,
        "times": times,
    }


def check_output(run: dict) -> dict:
    spec = SHAPES[run["shape_id"]]
    actual_flat = np.fromfile(run["output"], dtype=np.float32)
    expected_count = spec["M"] * spec["N"]
    if actual_flat.size != expected_count:
        raise ValueError(
            f"输出元素数错误：expected={expected_count}, "
            f"actual={actual_flat.size}, path={run['output']}"
        )
    actual = actual_flat.reshape(spec["M"], spec["N"])

    golden = np.load(INPUT_ROOT / run["shape_id"] / "C_golden.npy")
    if golden.shape != actual.shape:
        raise ValueError(f"Golden Shape 错误：expected={actual.shape}, actual={golden.shape}")

    diff = actual - golden
    abs_error = np.abs(diff)
    max_abs = float(abs_error.max())
    rmse = float(np.sqrt(np.mean(diff.astype(np.float64) ** 2)))
    golden_rms = float(np.sqrt(np.mean(golden.astype(np.float64) ** 2)))
    nrmse = rmse / max(golden_rms, 1e-12)
    golden_max = float(np.max(np.abs(golden)))
    tolerance = spec["atol"] + spec["rtol"] * np.abs(golden)
    tolerance_excess = abs_error - tolerance
    worst = np.unravel_index(np.argmax(tolerance_excess), abs_error.shape)
    sentinel_count = int(np.count_nonzero(actual == 123.0))
    zero_count = int(np.count_nonzero(actual == 0.0))
    finite_ok = bool(np.all(np.isfinite(actual)))
    elementwise_ok = bool(np.allclose(actual, golden, atol=spec["atol"], rtol=spec["rtol"]))
    passed = bool(finite_ok and elementwise_ok and nrmse <= 0.05 and sentinel_count == 0)

    return {
        "max_abs_error": max_abs,
        "rmse": rmse,
        "nrmse": nrmse,
        "golden_max_abs": golden_max,
        "tolerance_at_worst": float(tolerance[worst]),
        "max_tolerance_excess": float(tolerance_excess[worst]),
        "finite_ok": finite_ok,
        "elementwise_ok": elementwise_ok,
        "worst_index": tuple(int(x) for x in worst),
        "actual_at_worst": float(actual[worst]),
        "golden_at_worst": float(golden[worst]),
        "actual_min": float(actual.min()),
        "actual_max": float(actual.max()),
        "actual_mean": float(actual.mean()),
        "sentinel_count": sentinel_count,
        "zero_count": zero_count,
        "element_count": int(actual.size),
        "actual_head": actual.reshape(-1)[:16].tolist(),
        "passed": passed,
    }


def summarize_times(path: Path, version: str) -> dict:
    frame = pd.read_csv(path)
    required_columns = {"timing_schema", "launch_mode", "label", "iteration", "time_us"}
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"计时文件缺少列：{sorted(missing_columns)}")
    if set(frame["timing_schema"]) != {TIMING_SCHEMA}:
        raise ValueError(f"计时文件 schema 不匹配：{path}")
    if set(frame["launch_mode"]) != {RUNNER_MODE}:
        raise ValueError(f"计时文件启动方式不匹配：{path}")
    if set(frame["label"]) != {version}:
        raise ValueError(f"计时文件版本不匹配：{path}")

    values = frame["time_us"].to_numpy(dtype=np.float64)
    if values.size == 0:
        raise ValueError(f"计时文件没有样本：{path}")
    if not np.all(np.isfinite(values)) or np.any(values <= 0.0):
        raise ValueError(f"计时样本必须为有限正数：{path}")
    return {
        "median_us": float(np.median(values)),
        "mean_us": float(np.mean(values)),
        "p90_us": float(np.percentile(values, 90)),
        "std_us": float(np.std(values)),
        "cv": float(np.std(values) / max(np.mean(values), 1e-12)),
    }


def benchmark_variant(
    *,
    version: str,
    shape_id: str,
    tiling_id: str,
    buffer_num: int,
    warmup: int = VALIDATION_WARMUP,
    repeat: int = VALIDATION_REPEAT,
) -> pd.DataFrame:
    candidate = get_candidate(tiling_id)
    spec = SHAPES[shape_id]
    estimate = estimate_candidate(spec, candidate, buffer_num)
    if not estimate["valid"]:
        raise ValueError(f"{shape_id} 不支持 {tiling_id}")

    compile_runner(tiling_id, buffer_num)
    run = run_variant(
        version=version,
        shape_id=shape_id,
        tiling_id=tiling_id,
        buffer_num=buffer_num,
        warmup=warmup,
        repeat=repeat,
    )
    correctness = check_output(run)
    if not correctness["passed"]:
        raise AssertionError(
            {
                "version": version,
                "shape_id": shape_id,
                "tiling_id": tiling_id,
                "buffer_num": buffer_num,
                **correctness,
            }
        )

    timing = summarize_times(run["times"], version)
    flop_count = 2.0 * spec["M"] * spec["K"] * spec["N"]
    gflops = flop_count / max(timing["median_us"], 1e-12) / 1e3
    row = {
        "version": version,
        "runner_mode": RUNNER_MODE,
        "timing_schema": TIMING_SCHEMA,
        "shape_id": shape_id,
        "M": spec["M"],
        "K": spec["K"],
        "N": spec["N"],
        **estimate,
        **correctness,
        **timing,
        "gflops": float(gflops),
        "warmup": warmup,
        "repeat": repeat,
    }
    return pd.DataFrame([row])


### 11. 分阶段验证正确性

验证分三级进行：

- Level 1：4 个 K Tile 测试；
- Level 2：在 Level 1 上增加 2 个多输出 Tile 测试；
- Level 3：再增加 Core-stride 测试。

保持 `ENABLE_DOUBLE_BUFFER_VALIDATION = False`，先完成 Single Buffer 的 Level 1 到 Level 3。补全 TODO 5、6 后，将开关设为 `True` 并重新运行 Level 3。此时共有 7 个 Shape × 2 种 Buffer 模式，即 14 组测试。预期每组结果都满足 `passed = True` 且 `sentinel_count = 0`。


In [ ]:
VALIDATION_LEVELS = {
    1: ["K_FIRST_ONLY", "K_SECOND_ONLY", "K_BOTH_SUM", "K_RANDOM"],
    2: [
        "K_FIRST_ONLY",
        "K_SECOND_ONLY",
        "K_BOTH_SUM",
        "K_RANDOM",
        "MULTI_TILE_IDENTITY",
        "MULTI_TILE_RANDOM",
    ],
    3: [
        "K_FIRST_ONLY",
        "K_SECOND_ONLY",
        "K_BOTH_SUM",
        "K_RANDOM",
        "MULTI_TILE_IDENTITY",
        "MULTI_TILE_RANDOM",
        "CORE_STRIDE_RANDOM",
    ],
}

if VALIDATION_LEVEL not in VALIDATION_LEVELS:
    raise ValueError("VALIDATION_LEVEL must be 1, 2 or 3")

VALIDATION_SHAPE_IDS = VALIDATION_LEVELS[VALIDATION_LEVEL]
validation_rows = []

print("Validation level:", VALIDATION_LEVEL)
print("Validation shapes:", VALIDATION_SHAPE_IDS)
print("Buffer modes:", BUFFER_NUMS)

for buffer_num in BUFFER_NUMS:
    print("\n========================================")
    print(f"STUDENT DIRECT-KERNEL VALIDATION: T4 / BUFFER_NUM={buffer_num}")
    print("========================================")

    compile_runner(tiling_id="T4", buffer_num=buffer_num)

    for shape_id in VALIDATION_SHAPE_IDS:
        spec = SHAPES[shape_id]
        estimate = estimate_candidate(spec, get_candidate("T4"), buffer_num)
        assert estimate["valid"], {"shape_id": shape_id, "estimate": estimate}

        print("\n----------", shape_id, "----------")
        run = run_variant(
            version=VALIDATION_VERSION,
            shape_id=shape_id,
            tiling_id="T4",
            buffer_num=buffer_num,
            warmup=VALIDATION_WARMUP,
            repeat=VALIDATION_REPEAT,
        )
        correctness = check_output(run)
        timing = summarize_times(run["times"], VALIDATION_VERSION)
        row = {
            "version": VALIDATION_VERSION,
            "runner_mode": RUNNER_MODE,
            "timing_schema": TIMING_SCHEMA,
            "shape_id": shape_id,
            "buffer_num": buffer_num,
            **estimate,
            **correctness,
            **timing,
        }
        validation_rows.append(row)

        if not correctness["passed"]:
            raise AssertionError(
                {
                    "version": VALIDATION_VERSION,
                    "validation_level": VALIDATION_LEVEL,
                    "shape_id": shape_id,
                    "tiling_id": "T4",
                    "buffer_num": buffer_num,
                    **correctness,
                }
            )

validation_results = pd.DataFrame(validation_rows)
validation_csv = RESULT_ROOT / "student_direct_kernel_v2_0_validation.csv"
validation_results.to_csv(validation_csv, index=False)

display(
    validation_results[
        [
            "shape_id",
            "buffer_num",
            "mTiles",
            "nTiles",
            "kTiles",
            "outputTileCount",
            "max_abs_error",
            "rmse",
            "nrmse",
            "sentinel_count",
            "median_us",
            "passed",
        ]
    ]
)

assert validation_results["passed"].all()
print("\n当前等级的正确性验证全部通过。")
print("Saved:", validation_csv)


### 12. 运行 T1–T5 性能实验

完整 Sweep 前设置：

```python
VALIDATION_LEVEL = 3
ENABLE_DOUBLE_BUFFER_VALIDATION = True
RUN_FULL_SWEEP = True
```

实验包含 5 组 Tiling × 2 种 Buffer 模式，共 10 组配置。这里的 `BUFFER_NUM = 1 / 2` 分别表示 Single Buffer 和 Double Buffer，与存储层级 B1 无关。修改开关后，重新运行初始化单元、本阶段及后续分析单元。

`RESUME_SWEEP` 默认为 `False`，避免 Kernel、Shape、Tiling 或启动方式变化后误用旧 CSV。需要断点续跑时，将它改为 `True`，并保持 `SWEEP_VERSION`、`RUNNER_MODE` 和 `TIMING_SCHEMA` 不变。


In [ ]:
SWEEP_SHAPE_ID = "MAIN_BENCH"
EXPECTED_SWEEP_KEYS = {
    (candidate["id"], buffer_num) for candidate in TILING_CANDIDATES for buffer_num in BUFFER_NUMS
}

if RUN_FULL_SWEEP:
    assert VALIDATION_LEVEL == 3, "Set VALIDATION_LEVEL=3 before the sweep."
    assert ENABLE_DOUBLE_BUFFER_VALIDATION, "请先完成并通过 Double Buffer 正确性验证。"

SWEEP_CSV = RESULT_ROOT / "student_direct_kernel_v2_0_sweep.csv"

if RESUME_SWEEP and SWEEP_CSV.is_file():
    sweep_results = pd.read_csv(SWEEP_CSV)
    required_columns = {
        "version",
        "runner_mode",
        "timing_schema",
        "shape_id",
        "tiling_id",
        "buffer_num",
        "passed",
    }
    missing_columns = required_columns - set(sweep_results.columns)
    if missing_columns:
        raise ValueError(f"Sweep CSV 缺少列：{sorted(missing_columns)}")
    passed_mask = sweep_results["passed"].astype(str).str.lower() == "true"
    sweep_results = sweep_results[
        (sweep_results["version"] == SWEEP_VERSION)
        & (sweep_results["runner_mode"] == RUNNER_MODE)
        & (sweep_results["timing_schema"] == TIMING_SCHEMA)
        & (sweep_results["shape_id"] == SWEEP_SHAPE_ID)
        & passed_mask
    ].copy()
else:
    sweep_results = pd.DataFrame()

completed = set()
if not sweep_results.empty:
    completed = {
        (str(row["tiling_id"]), int(row["buffer_num"])) for _, row in sweep_results.iterrows()
    }

print("Existing completed variants:", sorted(completed))

if RUN_FULL_SWEEP:
    for candidate in TILING_CANDIDATES:
        tiling_id = candidate["id"]
        for buffer_num in BUFFER_NUMS:
            key = (tiling_id, buffer_num)
            if key in completed:
                print("Skip completed:", key)
                continue

            print("\n========================================")
            print("SWEEP:", tiling_id, f"BUFFER_NUM={buffer_num}")
            print("========================================")
            frame = benchmark_variant(
                version=SWEEP_VERSION,
                shape_id=SWEEP_SHAPE_ID,
                tiling_id=tiling_id,
                buffer_num=buffer_num,
                warmup=SWEEP_WARMUP,
                repeat=SWEEP_REPEAT,
            )
            sweep_results = pd.concat([sweep_results, frame], ignore_index=True)
            sweep_results = (
                sweep_results.sort_values(["tiling_id", "buffer_num"])
                .drop_duplicates(
                    subset=["version", "runner_mode", "timing_schema", "shape_id", "tiling_id", "buffer_num"],
                    keep="last",
                )
                .reset_index(drop=True)
            )
            sweep_results.to_csv(SWEEP_CSV, index=False)
            completed.add(key)
            display(
                frame[
                    [
                        "tiling_id",
                        "buffer_num",
                        "baseM",
                        "baseN",
                        "baseK",
                        "kTiles",
                        "median_us",
                        "p90_us",
                        "cv",
                        "gflops",
                        "max_abs_error",
                        "passed",
                    ]
                ]
            )
else:
    print("RUN_FULL_SWEEP=False：跳过完整性能 Sweep。")

if RUN_FULL_SWEEP:
    assert completed == EXPECTED_SWEEP_KEYS, {
        "expected": sorted(EXPECTED_SWEEP_KEYS),
        "completed": sorted(completed),
    }
    assert sweep_results["passed"].all()

print("Sweep CSV:", SWEEP_CSV)


### 13. 分析性能结果

Double Buffer 加速比定义为

$$
\mathrm{Speedup}_{DB}=\frac{T_{Single}}{T_{Double}}.
$$

结果大于 1 表示 Double Buffer 更快。Tile 大小、搬运量、`Mmad` 粒度和片上 Buffer 占用都会影响流水重叠，因此不同 Tiling 的加速比可能不同。仅比较具有相同 `direct_kernel_v2_0` 计时标识的记录。


In [ ]:
if sweep_results.empty:
    print("尚无 Sweep 结果。")
else:
    ordered = sweep_results.sort_values("median_us").reset_index(drop=True)
    display(
        ordered[
            [
                "tiling_id",
                "buffer_num",
                "baseM",
                "baseN",
                "baseK",
                "mTiles",
                "nTiles",
                "kTiles",
                "outputTileCount",
                "median_us",
                "p90_us",
                "cv",
                "gflops",
                "max_abs_error",
            ]
        ]
    )

    best = ordered.iloc[0]
    print("\nBest configuration:")
    print(
        {
            "tiling_id": best["tiling_id"],
            "buffer_num": int(best["buffer_num"]),
            "baseM": int(best["baseM"]),
            "baseN": int(best["baseN"]),
            "baseK": int(best["baseK"]),
            "median_us": float(best["median_us"]),
            "gflops": float(best["gflops"]),
        }
    )

    single = sweep_results[sweep_results["buffer_num"] == 1][
        ["tiling_id", "median_us", "gflops"]
    ].rename(columns={"median_us": "single_us", "gflops": "single_gflops"})
    double = sweep_results[sweep_results["buffer_num"] == 2][
        ["tiling_id", "median_us", "gflops"]
    ].rename(columns={"median_us": "double_us", "gflops": "double_gflops"})
    speedup_table = single.merge(double, on="tiling_id", how="inner")
    speedup_table["double_buffer_speedup"] = speedup_table["single_us"] / speedup_table["double_us"]
    speedup_table = speedup_table.sort_values("double_buffer_speedup", ascending=False).reset_index(
        drop=True
    )
    display(speedup_table)

    speedup_csv = RESULT_ROOT / "student_direct_kernel_v2_0_double_buffer_speedup.csv"
    speedup_table.to_csv(speedup_csv, index=False)
    summary = {
        "runner_mode": RUNNER_MODE,
        "timing_schema": TIMING_SCHEMA,
        "best": {
            "tiling_id": str(best["tiling_id"]),
            "buffer_num": int(best["buffer_num"]),
            "baseM": int(best["baseM"]),
            "baseN": int(best["baseN"]),
            "baseK": int(best["baseK"]),
            "median_us": float(best["median_us"]),
            "p90_us": float(best["p90_us"]),
            "gflops": float(best["gflops"]),
        },
        "validation_csv": str(RESULT_ROOT / "student_direct_kernel_v2_0_validation.csv"),
        "sweep_csv": str(SWEEP_CSV),
        "speedup_csv": str(speedup_csv),
    }
    summary_path = RESULT_ROOT / "student_direct_kernel_v2_0_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved summary:", summary_path)


## 实验总结

本实验把 MatMul 按 M、N、K 三个维度切成 Tile，在 Host 侧计算 Tile 数量和使用核数，再通过内核调用符将 GM 指针与基础参数传给 Kernel，完成输出 Tile 映射、GM 搬运和 K 维累加。分阶段输入用于定位偏移与累加错误；性能 Sweep 则在固定 Tiling 的前提下比较 Single Buffer 和 Double Buffer，避免把 Tiling 差异混入加速比。

## 实验扩展

1. 为什么输出 Tile 数量是 `mTiles * nTiles`，而不乘 `kTiles`？
2. 一维 `outputTile` 如何映射为 `(mTile, nTile)`？多核如何覆盖全部输出 Tile？
3. A、B、C 在 GM(ND) 中的 Offset 分别如何计算？
4. 第一块 K Tile 与后续 K Tile 的 `Mmad` 累加方式有什么不同？
5. 为什么 B 按 16 列 Fragment 从 B1(NZ) 转换为 B2(ZN)？
6. Double Buffer 为什么要在进入 K 循环前准备两个 K Tile？
7. 为什么稳态阶段预取 `kTile + 2`，而不是当前 Tile 或 `kTile + 1`？
8. 如何分别比较 Single Buffer 和 Double Buffer 下 T1–T5 的性能？哪个配置最快？
9. Double Buffer 是否对每一种 Tiling 都能提速？如何计算并解释加速比？
10. 最快配置是否一定占用最多片上 Buffer？为什么？

比较不同 Tiling 时固定 `buffer_num`；比较 Single Buffer 与 Double Buffer 时固定 `tiling_id`。运行下一单元查看参考答案。


In [ ]:
!cat answer/thought_questions.txt

## 参考答案

六处 TODO 的完整实现位于 `answer/05.01_answer/`。下面的单元分别显示 Host 侧和 Kernel 侧源码，最后保留复制到实验目录的命令。


In [ ]:
!cat ./answer/05.01_answer/op_host/cube_matmul_custom.cpp

In [ ]:
!cat ./answer/05.01_answer/op_kernel/cube_matmul_custom.cpp

如果要用参考实现继续实验，执行下一单元覆盖实验目录中的 Host 和 Kernel 文件，然后重新编译 Runner 并运行分阶段验证。


In [ ]:
!cp answer/05.01_answer/op_host/cube_matmul_custom.cpp Sources/L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student/op_host/cube_matmul_custom.cpp
!cp answer/05.01_answer/op_kernel/cube_matmul_custom.cpp Sources/L05_03_AscendC_Matmul_Tiling_DoubleBuffer_student/op_kernel/cube_matmul_custom.cpp
